In [66]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [67]:
df_video = pd.read_csv('videos.csv')
df_creator= pd.read_csv('creators.csv')

In [68]:
df_video.columns
df_video.drop(columns=['VQSCORE',
       'BITRATE', 'CATEGORY_TYPE', 'TITLE', 'DESC', 'MUSIC_TITLE',
       'MUSIC_AUTHOR', 'SNAPSHOT_TIME', 'RAW_JSON', 'MUSIC_PLAY_URL'], inplace=True)


In [69]:
df_video.columns

Index(['VIDEO_ID', 'CREATOR_ID', 'CREATE_TIME', 'ANCHOR_TYPES', 'VIEW_COUNT',
       'LIKE_COUNT', 'COMMENT_COUNT', 'SHARE_COUNT', 'SAVE_COUNT'],
      dtype='str')

In [70]:
df_video

,VIDEO_ID,CREATOR_ID,CREATE_TIME,ANCHOR_TYPES,VIEW_COUNT,LIKE_COUNT,COMMENT_COUNT,SHARE_COUNT,SAVE_COUNT
0,7574282336460737800,endersff,2026-01-09 15:03:48.0000000,NaN,11600000,45700,202,1074,1816
1,7583972786322885889,aigonew1ld,2025-12-28 15:20:00.0000000,NaN,16300000,909700,9759,759100,74587
2,7585089116388379911,khucanhhung,2025-12-28 20:00:00.0000000,NaN,268400,7964,158,2353,149
3,7585469260718296328,endersff,2025-12-25 16:35:31.0000000,"[33, 54]",10700,78,9,5,6
4,7585789833734737172,hieuride17x,2026-01-01 00:00:00.0000000,NaN,143200,22300,101,3764,1565
...,...,...,...,...,...,...,...,...,...
68251,7620852701940698389,vtn.10,2026-03-24 23:00:46.0000000,NaN,147,16,0,1,0
68252,7620853672796294420,ziuen19_11,2026-03-24 23:04:35.0000000,NaN,1159,227,6,73,19
68253,7620853998379175189,stevent107,2026-03-24 23:05:52.0000000,[33],31,1,0,0,0
68254,7620855160511712533,simontu89,2026-03-24 23:10:22.0000000,NaN,0,0,0,1,0


In [71]:
df_video['CREATE_TIME'] = pd.to_datetime(df_video['CREATE_TIME'], errors='coerce')
crawl_time = '2026-03-24 23:55:44' 
crawl_time = pd.to_datetime(crawl_time)

In [72]:
# Bóc tách Ngày trong tuần và Khung giờ từ CREATE_TIME
df_video['DAY_OF_WEEK'] = df_video['CREATE_TIME'].dt.day_name()
df_video['HOUR'] = df_video['CREATE_TIME'].dt.hour


In [73]:
# Khởi tạo một DataFrame rỗng, chỉ chứa cột CREATOR_ID để đắp dần dữ liệu vào
creator_features = pd.DataFrame({'CREATOR_ID': df_video['CREATOR_ID'].unique()})

In [74]:
# Hàm tìm giá trị xuất hiện nhiều nhất (Mode) cho Ngày/Giờ đăng
def get_mode(x):
    m = x.mode()
    return m.iloc[0] if not m.empty else np.nan

g1_stats = df_video.groupby('CREATOR_ID').agg(
    RECENT_VIDEO_COUNT=('VIDEO_ID', 'count'),          # a: Tổng video
    FIRST_POST=('CREATE_TIME', 'min'),                 # Ngày bắt đầu
    LAST_POST=('CREATE_TIME', 'max'),                  # Ngày cuối đăng bài
    MOST_ACTIVE_DAY=('DAY_OF_WEEK', get_mode),         # Ngày đăng nhiều nhất
    MOST_ACTIVE_HOUR=('HOUR', get_mode)                # Giờ đăng nhiều nhất
).reset_index()

In [75]:
# DAYS_SINCE_LAST_POST (Độ trễ từ lần đăng cuối)
g1_stats['DAYS_SINCE_LAST_POST'] = (crawl_time - g1_stats['LAST_POST']).dt.days

# VIDEOS_PER_WEEK = Tổng số video (a) / Số tuần hoạt động (b)
active_weeks = ((g1_stats['LAST_POST'] - g1_stats['FIRST_POST']).dt.days / 7).clip(lower=1)
g1_stats['VIDEOS_PER_WEEK'] = g1_stats['RECENT_VIDEO_COUNT'] / active_weeks

# Gộp vào bảng chính (Chỉ lấy các cột cần thiết)
cols_g1 = ['CREATOR_ID', 'RECENT_VIDEO_COUNT', 'VIDEOS_PER_WEEK', 'DAYS_SINCE_LAST_POST', 'MOST_ACTIVE_DAY', 'MOST_ACTIVE_HOUR']
creator_features = creator_features.merge(g1_stats[cols_g1], on='CREATOR_ID', how='left')

In [76]:
creator_features

,CREATOR_ID,RECENT_VIDEO_COUNT,VIDEOS_PER_WEEK,DAYS_SINCE_LAST_POST,MOST_ACTIVE_DAY,MOST_ACTIVE_HOUR
0,endersff,24,1.887640,0,Monday,10
1,aigonew1ld,68,5.600000,0,Monday,18
2,khucanhhung,33,3.500000,19,Tuesday,20
3,hieuride17x,27,2.123596,0,Thursday,19
4,wonrtoys,21,1.792683,7,Tuesday,16
...,...,...,...,...,...,...
943,muoiphim,4,2.800000,7,Saturday,11
944,_nlbaotran,467,217.933333,0,Wednesday,21
945,ntpmm.ent,35,15.312500,0,Tuesday,20
946,h1_.sleepybeauty,2,2.000000,6,Tuesday,14


In [77]:
# Tính thẳng Trung Vị (Median) của toàn bộ các chỉ số
g2_stats = df_video.groupby('CREATOR_ID').agg(
    MEDIAN_VIEWS_3M=('VIEW_COUNT', 'median'),
    MEDIAN_LIKES_3M=('LIKE_COUNT', 'median'),
    MEDIAN_COMMENTS_3M=('COMMENT_COUNT', 'median'),
    MEDIAN_SHARES_3M=('SHARE_COUNT', 'median'),
    MEDIAN_SAVES_3M=('SAVE_COUNT', 'median')
).reset_index()

creator_features = creator_features.merge(g2_stats, on='CREATOR_ID', how='left')

In [78]:
creator_features

,CREATOR_ID,RECENT_VIDEO_COUNT,VIDEOS_PER_WEEK,DAYS_SINCE_LAST_POST,MOST_ACTIVE_DAY,MOST_ACTIVE_HOUR,MEDIAN_VIEWS_3M,MEDIAN_LIKES_3M,MEDIAN_COMMENTS_3M,MEDIAN_SHARES_3M,MEDIAN_SAVES_3M
0,endersff,24,1.887640,0,Monday,10,45950.0,1544.0,21.0,16.5,99.5
1,aigonew1ld,68,5.600000,0,Monday,18,17950.0,661.0,10.0,331.0,72.0
2,khucanhhung,33,3.500000,19,Tuesday,20,6946.0,25.0,3.0,1.0,2.0
3,hieuride17x,27,2.123596,0,Thursday,19,69500.0,7486.0,70.0,558.0,764.0
4,wonrtoys,21,1.792683,7,Tuesday,16,149700.0,1843.0,212.0,84.0,134.0
...,...,...,...,...,...,...,...,...,...,...,...
943,muoiphim,4,2.800000,7,Saturday,11,257.0,10.0,0.0,0.0,0.0
944,_nlbaotran,467,217.933333,0,Wednesday,21,7831.0,1181.0,4.0,22.0,94.0
945,ntpmm.ent,35,15.312500,0,Tuesday,20,26200.0,1957.0,32.0,297.0,44.0
946,h1_.sleepybeauty,2,2.000000,6,Tuesday,14,1645.5,175.5,3.5,2.5,5.5


In [79]:
# ===================================================================
print("Đang xử lý Nhóm 3...")

# Tính TỔNG của tất cả để làm công thức chia: Sum(Chỉ số) / Sum(View)
g3_stats = df_video.groupby('CREATOR_ID').agg(
    SUM_VIEWS=('VIEW_COUNT', 'sum'),
    SUM_LIKES=('LIKE_COUNT', 'sum'),
    SUM_COMMENTS=('COMMENT_COUNT', 'sum'),
    SUM_SHARES=('SHARE_COUNT', 'sum'),
    SUM_SAVES=('SAVE_COUNT', 'sum')
).reset_index()

# Xử lý tránh lỗi chia cho 0
g3_stats['SUM_VIEWS'] = g3_stats['SUM_VIEWS'].replace(0, np.nan)

# Tính Rate
g3_stats['LIKE_RATE'] = g3_stats['SUM_LIKES'] / g3_stats['SUM_VIEWS']
g3_stats['COMMENT_RATE'] = g3_stats['SUM_COMMENTS'] / g3_stats['SUM_VIEWS']
g3_stats['SHARE_RATE'] = g3_stats['SUM_SHARES'] / g3_stats['SUM_VIEWS']
g3_stats['SAVE_RATE'] = g3_stats['SUM_SAVES'] / g3_stats['SUM_VIEWS']

cols_g3 = ['CREATOR_ID', 'LIKE_RATE', 'COMMENT_RATE', 'SHARE_RATE', 'SAVE_RATE']
creator_features = creator_features.merge(g3_stats[cols_g3], on='CREATOR_ID', how='left')

Đang xử lý Nhóm 3...


In [80]:
creator_features

,CREATOR_ID,RECENT_VIDEO_COUNT,VIDEOS_PER_WEEK,DAYS_SINCE_LAST_POST,MOST_ACTIVE_DAY,MOST_ACTIVE_HOUR,MEDIAN_VIEWS_3M,MEDIAN_LIKES_3M,MEDIAN_COMMENTS_3M,MEDIAN_SHARES_3M,MEDIAN_SAVES_3M,LIKE_RATE,COMMENT_RATE,SHARE_RATE,SAVE_RATE
0,endersff,24,1.887640,0,Monday,10,45950.0,1544.0,21.0,16.5,99.5,0.005932,0.000055,0.000127,0.000296
1,aigonew1ld,68,5.600000,0,Monday,18,17950.0,661.0,10.0,331.0,72.0,0.045475,0.000615,0.042657,0.003719
2,khucanhhung,33,3.500000,19,Tuesday,20,6946.0,25.0,3.0,1.0,2.0,0.032312,0.000921,0.014756,0.001796
3,hieuride17x,27,2.123596,0,Thursday,19,69500.0,7486.0,70.0,558.0,764.0,0.118109,0.000885,0.016626,0.012095
4,wonrtoys,21,1.792683,7,Tuesday,16,149700.0,1843.0,212.0,84.0,134.0,0.013115,0.001276,0.000491,0.000829
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
943,muoiphim,4,2.800000,7,Saturday,11,257.0,10.0,0.0,0.0,0.0,0.017356,0.000510,0.000000,0.000510
944,_nlbaotran,467,217.933333,0,Wednesday,21,7831.0,1181.0,4.0,22.0,94.0,0.195444,0.000637,0.004708,0.016457
945,ntpmm.ent,35,15.312500,0,Tuesday,20,26200.0,1957.0,32.0,297.0,44.0,0.072782,0.000969,0.015884,0.001921
946,h1_.sleepybeauty,2,2.000000,6,Tuesday,14,1645.5,175.5,3.5,2.5,5.5,0.106655,0.002127,0.001519,0.003342


In [ ]:
# ===================================================================
# NHÓM 4: ĐỘ ỔN ĐỊNH & BẤP BÊNH (Consistency vs. Volatility)
# ===================================================================
print("Đang xử lý Nhóm 4...")

# VIEW_STD_DEV (Độ lệch chuẩn)
g4_stats = df_video.groupby('CREATOR_ID').agg(
    VIEW_STD_DEV=('VIEW_COUNT', 'std')
).reset_index()
g4_stats['VIEW_STD_DEV'] = g4_stats['VIEW_STD_DEV'].fillna(0) # Điền 0 nếu chỉ có 1 video

# VIRALITY_INDEX: Cần join lại với Median View để lấy mốc so sánh
df_videos_merged = df_video.merge(g2_stats[['CREATOR_ID', 'MEDIAN_VIEWS_3M']], on='CREATOR_ID', how='left')

# Điều kiện: Vượt 1 Triệu View HOẶC Vượt gấp 10 lần mức Trung Vị của chính họ
df_videos_merged['IS_VIRAL'] = (
    (df_videos_merged['VIEW_COUNT'] >= 1000000) | 
    (df_videos_merged['VIEW_COUNT'] > 10 * df_videos_merged['MEDIAN_VIEWS_3M'])
).astype(int)


 # CẦN ĐỊNH NGHĨA VIDEO VIRAL
# Đếm tổng số video Viral
# virality_counts = df_videos_merged.groupby('CREATOR_ID')['IS_VIRAL'].sum().reset_index()
# virality_counts.rename(columns={'IS_VIRAL': 'VIRALITY_INDEX'}, inplace=True)

# Gộp vào bảng chính
creator_features = creator_features.merge(g4_stats, on='CREATOR_ID', how='left')
# creator_features = creator_features.merge(virality_counts, on='CREATOR_ID', how='left')

Đang xử lý Nhóm 4...


In [82]:
creator_features

,CREATOR_ID,RECENT_VIDEO_COUNT,VIDEOS_PER_WEEK,DAYS_SINCE_LAST_POST,MOST_ACTIVE_DAY,MOST_ACTIVE_HOUR,MEDIAN_VIEWS_3M,MEDIAN_LIKES_3M,MEDIAN_COMMENTS_3M,MEDIAN_SHARES_3M,MEDIAN_SAVES_3M,LIKE_RATE,COMMENT_RATE,SHARE_RATE,SAVE_RATE,VIEW_STD_DEV,VIRALITY_INDEX
0,endersff,24,1.887640,0,Monday,10,45950.0,1544.0,21.0,16.5,99.5,0.005932,0.000055,0.000127,0.000296,2.358877e+06,1
1,aigonew1ld,68,5.600000,0,Monday,18,17950.0,661.0,10.0,331.0,72.0,0.045475,0.000615,0.042657,0.003719,2.262297e+06,12
2,khucanhhung,33,3.500000,19,Tuesday,20,6946.0,25.0,3.0,1.0,2.0,0.032312,0.000921,0.014756,0.001796,1.456088e+05,4
3,hieuride17x,27,2.123596,0,Thursday,19,69500.0,7486.0,70.0,558.0,764.0,0.118109,0.000885,0.016626,0.012095,2.057257e+05,2
4,wonrtoys,21,1.792683,7,Tuesday,16,149700.0,1843.0,212.0,84.0,134.0,0.013115,0.001276,0.000491,0.000829,2.803034e+05,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
943,muoiphim,4,2.800000,7,Saturday,11,257.0,10.0,0.0,0.0,0.0,0.017356,0.000510,0.000000,0.000510,5.216847e+02,0
944,_nlbaotran,467,217.933333,0,Wednesday,21,7831.0,1181.0,4.0,22.0,94.0,0.195444,0.000637,0.004708,0.016457,3.312671e+04,17
945,ntpmm.ent,35,15.312500,0,Tuesday,20,26200.0,1957.0,32.0,297.0,44.0,0.072782,0.000969,0.015884,0.001921,6.149219e+04,1
946,h1_.sleepybeauty,2,2.000000,6,Tuesday,14,1645.5,175.5,3.5,2.5,5.5,0.106655,0.002127,0.001519,0.003342,5.621499e+02,0
